In [1]:
from datasets import load_dataset
import numpy as np
import evaluate
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaModel, RobertaForSequenceClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "roberta-base"

c:\Users\lewy7\Documents\GitHub\roberta-hypernet-custom\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = RobertaTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=512)

dataset = load_dataset("glue", "cola")

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [3]:
# --- Custom LoRA Layer ---
class DynamicLoRALayer(nn.Module):
    def __init__(self, in_dim, r, hypernet):
        super().__init__()
        self.in_dim = in_dim
        self.r = r
        self.hypernet = hypernet
        self.training_mode = True
        self.A_fixed = None
        self.B_fixed = None

    def set_eval_mode(self):
        # Called once before evaluation
        self.training_mode = False
        with torch.no_grad():
            self.A_fixed = torch.randn(self.r, self.in_dim).to(next(self.parameters()).device)
            self.B_fixed = self.hypernet(self.A_fixed)

    def forward(self, x):
        if self.training_mode:
            A = torch.randn(self.r, self.in_dim).to(x.device)
            B = self.hypernet(A)
        else:
            A = self.A_fixed
            B = self.B_fixed

        # Apply low-rank adapter: x + (x @ B.T @ A)
        lora_out = torch.matmul(torch.matmul(x, B.T), A)
        return x + lora_out

In [4]:
# --- Hypernetwork: A -> B ---
class LoRAHyperNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, lora_dim):
        super().__init__()
        self.fc1 = nn.Linear(lora_dim * input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, input_dim * lora_dim)

    def forward(self, A):
        flat = A.view(-1)
        h = torch.relu(self.fc1(flat))
        B = self.fc2(h).view(A.size(1), A.size(0))  # B shape: [in_dim, r]
        return B

In [ ]:
class RobertaWithDynamicLoRA(RobertaForSequenceClassification):
    def __init__(self, config, lora_r=8, hypernet_hidden_dim=512):
        super().__init__(config)
        self.config.output_hidden_states = True 
        self.hidden_size = config.hidden_size
        self.lora_r = lora_r

        # Create hypernetwork and dynamic LoRA layer
        self.hypernet = LoRAHyperNet(self.hidden_size, hypernet_hidden_dim, lora_r)
        self.lora = DynamicLoRALayer(self.hidden_size, lora_r, self.hypernet)

        # Inject after the last layer's output dense
        self.output_dense = self.roberta.encoder.layer[-1].output.dense
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
               
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )

        hidden_states = outputs.hidden_states[-1]  # [batch, seq_len, hidden_dim]
        _, _, hidden_dim = hidden_states.shape

        if self.training:
            # Sample A from normal distribution per step
            A = torch.randn((self.lora_r, hidden_dim), device=hidden_states.device)
        else:
            # For evaluation use fixed A LoRA matrix
            A = self.lora.A_fixed

        B = self.hypernet(A)  # [d_model, r]
        lora_weight = torch.matmul(B, A)  # [d_model, d_model]

        # Apply LoRA adaptation
        adapted_hidden = hidden_states + torch.matmul(hidden_states, lora_weight.T)

        # Use [CLS] token for classification
        cls_output = adapted_hidden[:, 0]  # [batch, hidden]
        logits = self.dropout(cls_output)
        logits = self.classifier.out_proj(logits)  

        loss = None
        if labels is not None:
            loss_fn = torch.nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"logits": logits, "loss": loss} if loss is not None else {"logits": logits}



In [6]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=8,
    lora_dropout=0.1
)

base_model = RobertaWithDynamicLoRA.from_pretrained(model_name)
base_model = get_peft_model(base_model, peft_config=peft_config)

total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Some weights of RobertaWithDynamicLoRA were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'hypernet.fc1.bias', 'hypernet.fc1.weight', 'hypernet.fc2.bias', 'hypernet.fc2.weight', 'lora.hypernet.fc1.bias', 'lora.hypernet.fc1.weight', 'lora.hypernet.fc2.bias', 'lora.hypernet.fc2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 131,832,324
Trainable parameters: 887,042


In [7]:
print(base_model.roberta.encoder.layer[-1].attention.self.query)

lora.Linear(
  (base_layer): Linear(in_features=768, out_features=768, bias=True)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.1, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=768, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=768, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


In [8]:
training_args = TrainingArguments(
    output_dir="./outputs/reoberta_base_cola",
    eval_strategy="epoch",
    # eval_steps=25,
    save_strategy="steps",
    save_steps=1000000,
    learning_rate=4e-4,
    per_device_train_batch_size=16, # 16
    gradient_accumulation_steps=2, # 2
    per_device_eval_batch_size=32,
    num_train_epochs=1, # 80
    logging_dir="./logs/roberta_base_cola",
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    disable_tqdm=False,
    # remove_unused_columns=False
)

In [9]:
from transformers import Trainer

class CustomTrainer(Trainer):
    def evaluate(self, *args, **kwargs):
        # Put your LoRA modules into eval mode before evaluation
        self.model.lora.set_eval_mode()
        
        return super().evaluate(*args, **kwargs)

In [10]:
trainer = CustomTrainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
